# Notebook 02 — ColPali Indexing + CLIP Baseline

Builds two retrieval indexes over the OpenI CXR images:
1. **ColPali** (primary) — late-interaction multimodal retrieval
2. **CLIP ViT-L/14** (baseline) — global embedding FAISS index

Saves both to Google Drive for persistence across sessions.

In [ ]:
!pip install -q colpali-engine byaldi
!pip install -q 'transformers>=4.45.0' accelerate
!pip install -q open-clip-torch faiss-gpu

In [ ]:
import os, sys
from google.colab import drive, userdata
drive.mount('/content/drive')

DRIVE_ROOT  = '/content/drive/MyDrive/cxr_rag'
IMAGES_DIR  = '/content/openi/images'  # must exist from notebook 01
REPO_PATH   = '/content/cxr-rag-system'
sys.path.insert(0, REPO_PATH)

HF_TOKEN = userdata.get('HF_TOKEN')  # set in Colab Secrets

In [ ]:
# ── 1. Build ColPali Index ────────────────────────────────────────────────────
# ColPali (~7 GB download, one-time)
# Treats each CXR image as a visual document and indexes via late-interaction

from src.retrieval.colpali_retriever import ColPaliRetriever

COLPALI_INDEX_DIR = os.path.join(DRIVE_ROOT, 'colpali_index')
os.makedirs(COLPALI_INDEX_DIR, exist_ok=True)

retriever = ColPaliRetriever()
retriever.build_index(images_dir=IMAGES_DIR, index_save_dir=COLPALI_INDEX_DIR)
print('ColPali index built and saved to Drive')

In [ ]:
# ── Test ColPali retrieval ────────────────────────────────────────────────────
results = retriever.search('pleural effusion bilateral', k=3)
print(f'Top-{len(results)} results for query: "pleural effusion bilateral"')

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(results), figsize=(12, 4))
for ax, r in zip(axes, results):
    ax.imshow(r['image'], cmap='gray')
    ax.set_title(f"Score: {r['score']:.3f}")
    ax.axis('off')
plt.suptitle('ColPali Top-3 Retrieved CXRs')
plt.tight_layout()
plt.show()

In [ ]:
# ── Free ColPali VRAM before loading CLIP ─────────────────────────────────────
import torch, gc
del retriever
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# ── 2. Build CLIP Index (Baseline) ───────────────────────────────────────────
import glob
from src.retrieval.clip_retriever import CLIPRetriever

CLIP_INDEX_DIR = os.path.join(DRIVE_ROOT, 'clip_index')
os.makedirs(CLIP_INDEX_DIR, exist_ok=True)

image_paths = glob.glob(os.path.join(IMAGES_DIR, '**/*.png'), recursive=True)
print(f'Found {len(image_paths)} images to index')

clip_retriever = CLIPRetriever()
clip_retriever.build_index(image_paths, batch_size=128)
clip_retriever.save_index(CLIP_INDEX_DIR)
print('CLIP index saved to Drive')

In [ ]:
# ── Test CLIP retrieval ───────────────────────────────────────────────────────
results_clip = clip_retriever.search_by_text('pleural effusion bilateral', k=3)

fig, axes = plt.subplots(1, len(results_clip), figsize=(12, 4))
for ax, r in zip(axes, results_clip):
    ax.imshow(r['image'], cmap='gray')
    ax.set_title(f"Score: {r['score']:.3f}")
    ax.axis('off')
plt.suptitle('CLIP Top-3 Retrieved CXRs')
plt.tight_layout()
plt.show()